In [3]:
from embedder import Embedder

embed = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embed.encode(query)

print(v[0])

-0.02058203437252893


In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
target = [doc for doc in documents if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"][0]

target_vector = embed.encode(target["content"])

similarity = v.dot(target_vector)
print(similarity)

0.36107027225589694


In [ ]:
from gitsource import chunk_documents
import numpy as np
from tqdm.auto import tqdm

chunks = chunk_documents(documents, size=2000, step=1000)

batch_size = 50
X = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    batch_vectors = embed.encode_batch([c["content"] for c in batch])
    X.extend(batch_vectors)

X = np.array(X)

scores = X.dot(v)
idx = np.argmax(scores)
print(chunks[idx]["filename"])

  0%|          | 0/6 [00:00<?, ?it/s]

02-vector-search/lessons/07-sqlitesearch-vector.md


In [7]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

query2 = "What metric do we use to evaluate a search engine?"
v_query2 = embed.encode(query2)

results = vindex.search(v_query2, num_results=5)
print(results[0]["filename"])

04-evaluation/lessons/05-search-metrics.md


In [8]:
from minsearch import Index

# build text search index
tindex = Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)

query3 = "How do I store vectors in PostgreSQL?"
v_query3 = embed.encode(query3)

# vector search top5
vector_results = vindex.search(v_query3, num_results=5)
vector_files = [r["filename"] for r in vector_results]

# text search top5
text_results = tindex.search(query3, num_results=5)
text_files = [r["filename"] for r in text_results]

print("Vector search results:")
for f in vector_files:
    print(f)

print("\nText search results:")
for f in text_files:
    print(f)

# find files only in vector but not in text
diff = set(vector_files) - set(text_files)
print("\nOnly in vector search:")
print(diff)

Vector search results:
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md

Text search results:
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md

Only in vector search:
{'02-vector-search/lessons/08-pgvector.md'}


In [9]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

query4 = "How do I give the model access to tools?"
v_query4 = embed.encode(query4)

vector_results4 = vindex.search(v_query4, num_results=5)
text_results4 = tindex.search(query4, num_results=5)

results = rrf([vector_results4, text_results4])
print(results[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md
